<a href="https://colab.research.google.com/github/MusaAlver/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaAlver/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Refresh / Content Opportunity Scoring**

I want to explore which existing pages should be reviewed first when their search performance shows signs of decline. Reviewing every page manually is not realistic, so a ranked list could help focus attention on pages that may deserve a closer look. I chose this lane because it connects directly to a useful decision: which pages should be reviewed first. I will use observable search and content signals and compare simple rules with data-driven scoring methods.

In [13]:
import os
import subprocess
import pandas as pd

REPO_DIR = "/content/flyrank-ml-internship"
REPO_URL = "https://github.com/MusaAlver/flyrank-ml-internship.git"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully.
Rows: 30000
Columns: 44


**Research question:** Which pages should be prioritized for content review based on observable search and content-performance signals?

The unit of analysis is one content page. The output will be a ranked list of pages that may deserve review. A content or SEO team could use this ranking to decide which pages to inspect and potentially refresh first.

A wrong recommendation could waste time on a page that did not need attention. On the other hand, missing a genuinely declining page could mean losing an opportunity to react early. Data can help because signals such as impressions, average position, CTR, content age, and update history may interact in ways that are difficult to capture with one fixed rule.

In [10]:
duplicate_pages = df["content_id"].duplicated().sum()

print("Unit of analysis: one content page")
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", duplicate_pages)

Unit of analysis: one content page
Unique content IDs: 30000
Duplicate content IDs: 0


The starter dataset gives enough variation to make this lane worth investigating. It contains a large number of content pages, and a substantial share of them are currently marked as declining. I also looked at the relationship between average search position and CTR for pages with enough impressions. These are descriptive observations from the starter data, not causal claims, but they suggest that page performance is varied enough to make prioritization useful.

In [11]:
declining_rate = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .mean()
)

visible = df[
    (df["impressions_90d"] >= 100) &
    (df["avg_position"] > 0)
].copy()

position_ctr_corr = visible["avg_position"].corr(visible["ctr"])

print(f"Total pages: {len(df):,}")
print(f"Pages currently marked as declining: {declining_rate:.1%}")
print(f"Pages with 100+ impressions and valid position data: {len(visible):,}")
print(f"Average position vs CTR correlation: {position_ctr_corr:.3f}")

Total pages: 30,000
Pages currently marked as declining: 54.2%
Pages with 100+ impressions and valid position data: 22,006
Average position vs CTR correlation: -0.239


This project can describe observed and directional relationships in the provided data and evaluate whether a scoring approach can help prioritize pages for review. The final output should be treated as decision support: it can suggest where a team may want to look first.

It cannot prove that a specific factor causes a Google ranking change, explain Google's ranking algorithm, or guarantee that refreshing a recommended page will improve performance. I will also avoid using outcome-derived fields such as trend_direction or trend_pct as model features when they would leak information about the target.

In [12]:
leakage_fields = ["trend_direction", "trend_pct"]

print("Fields that should not be used as predictive features for a decline target:")
for field in leakage_fields:
    print("-", field)

print("\nReason: they contain or directly determine information about the outcome.")

Fields that should not be used as predictive features for a decline target:
- trend_direction
- trend_pct

Reason: they contain or directly determine information about the outcome.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`